In [106]:
# =============================================================================
#  CONFIGURATION & CONSTANTS (Centralized)
#  Single source of truth untuk semua hyperparameter, paths, dan magic numbers
# =============================================================================

# ============== FILE I/O CONSTANTS ==============
DATASET_PATH = "rbsqli_dataset.csv"
MODEL_PATH = "model_sqli_nb.pkl"
CSV_USECOLS = [0, 1, 2]
CSV_DTYPES = {0: str, 1: str, 2: str}

# ============== PREPROCESSING & NLP CONSTANTS ==============
JUMLAH_JOBS = 8  # Parallel jobs untuk preprocessing (3 Physical Cores)

# Pre-compiled regex patterns untuk performa optimal
REGEX_SINGLE_QUOTE = r"'[^']*'"
REGEX_DOUBLE_QUOTE = r'"[^"]*"'
REGEX_DIGITS = r'\d+'
REGEX_TOKENS = r"[a-z0-9_]+|--|/\*|\*/|'|\"|\(|\)|=|<|>|;|#|,|\*|\+|-|%"

# Linguistic constants
STOPWORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for",
    "of", "and", "with", "by", "as", "be", "was", "are", "were",
    "this", "that", "have", "has", "had", "do", "does", "did",
    "but", "so", "if", "then", "than", "its", "into", "from",
    "there", "their", "they", "will", "would", "could", "should"
}

SQL_KEYWORDS = {
    "select", "from", "where", "and", "or", "not", "is", "in",
    "like", "union", "insert", "update", "delete", "drop", "create",
    "table", "into", "values", "order", "by", "group", "having",
    "join", "on", "null", "true", "false", "case", "when", "then",
    "else", "end", "limit", "offset", "between", "exists", "all",
    "distinct", "count", "sum", "max", "min", "avg", "sleep",
    "benchmark", "char", "concat", "substring", "load_file",
    "outfile", "exec", "execute", "cast", "convert", "if"
}

FUZZY_RATIO_THRESHOLD = 90  # Threshold untuk deteksi leakage

# ============== DATA BALANCING CONSTANTS ==============
TARGET_PER_KATEGORI = 15000  # Target baris per kategori SQLi
KATEGORI_TARGET = [
    'Error-Based', 'meta_based', 'stackqueries_based', 
    'Time-Based', 'Union-Based', 'boolean-based'
]

# ============== TRAIN-TEST SPLIT CONSTANTS ==============
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ============== MODEL HYPERPARAMETERS ==============
# TF-IDF Configuration
TFIDF_ANALYZER = "word"
TFIDF_NGRAM_RANGE = (1, 3)
TFIDF_MIN_DF = 2
TFIDF_MAX_FEATURES = 5000
TFIDF_SUBLINEAR_TF = True
TFIDF_TOKEN_PATTERN = r"\S+"

# Naive Bayes Configuration
NB_ALPHA = 0.1

# Cross-Validation Configuration
CV_N_SPLITS = 5

# ============== DISPLAY CONSTANTS ==============
SEPARATOR_LONG = "=" * 65
SEPARATOR_MEDIUM = "─" * 40
SEPARATOR_SHORT = "─" * 60
MAX_DISPLAY_LENGTH = 70
MAX_EXAMPLES = 5

# Classification targets untuk reporting
CLASSIFICATION_TARGETS = ["Normal (0)", "SQLI (1)"]

In [81]:
# =============================================================================
#  UTILITAS BERSAMA
#  Dipakai oleh tahap training, inferensi, dan export hasil
# =============================================================================

import os
import re
import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from rapidfuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

In [113]:
# =============================================================================
#  1. PRE-COMPILE REGEX PATTERNS (Cukup di-run sekali di luar fungsi)
# =============================================================================
# Menggunakan [^'] jauh lebih cepat daripada (.*?) karena menghilangkan backtracking
# Patterns sudah didefinisikan di CONFIGURATION cell, sekarang di-compile
REGEX_SINGLE_QUOTE = re.compile(REGEX_SINGLE_QUOTE)
REGEX_DOUBLE_QUOTE = re.compile(REGEX_DOUBLE_QUOTE)
REGEX_DIGITS = re.compile(REGEX_DIGITS)

# Gabungkan tokenisasi menjadi satu pattern solid
REGEX_TOKENS = re.compile(REGEX_TOKENS)


def preprocess_text(text):
    """
    Fungsi preprocessing teks versi optimasi tinggi.
    Menghapus redundansi proses dan mempercepat eksekusi regex & loop.
    """
    if not isinstance(text, str):
        return ""
    # 1. Normalisasi case di awal untuk menghindari multiple passes
    text = text.lower()

    # 2. Eksekusi Regex yang sudah di-compile
    text = REGEX_SINGLE_QUOTE.sub("'str'", text)
    text = REGEX_DOUBLE_QUOTE.sub('"str"', text)
    text = REGEX_DIGITS.sub('0', text)

    # 3. Tokenisasi cepat
    tokens = REGEX_TOKENS.findall(text)

    # 4. OPTIMASI LOOP: List Comprehension berbasis C-level
    # Logika ekkuivalen: Simpan jika dia adalah KEYWORD SQL ATAU dia BUKAN termasuk STOPWORDS
    filtered = [t for t in tokens if (t in SQL_KEYWORDS or t not in STOPWORDS)]

    return " ".join(filtered)

def proses_blok_kebocoran(test_chunk, train_list, threshold):
    """
    Memproses pengecekan kebocoran per blok data untuk meminimalkan overhead.
    Menggunakan C++ level loop dari rapidfuzz dengan fitur early exit.
    """
    hasil = []
    for q in test_chunk:
        # extractOne langsung mengeliminasi string yang panjangnya tidak ideal (length filtering)
        res = process.extractOne(
            q, 
            train_list, 
            scorer=fuzz.ratio, 
            score_cutoff=threshold
        )
        # Jika res tidak None, berarti ada kecocokan di atas threshold (Bocor)
        hasil.append(res is not None)
    return hasil



In [83]:
# =============================================================================
#  TAHAP 1 — MEMUAT DATASET (VERSI OPTIMAL & SUPER CEPAT)
# =============================================================================

print(SEPARATOR_LONG)
print("  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES")
print(SEPARATOR_LONG)

# OPTIMASI:
# 1. usecols=[0, 1, 2] -> Hanya ambil 3 kolom pertama langsung dari disk (menghemat RAM & I/O).
# 2. dtype=str -> Memaksa Pandas membaca data sebagai string tanpa menebak tipe data (menghemat CPU).
# 3. engine='c' -> Memastikan beralih ke parser bahasa C yang jauh lebih cepat dari parser Python.
df = pd.read_csv(
    DATASET_PATH,
    usecols=CSV_USECOLS,
    dtype=CSV_DTYPES,
    engine='c',
    low_memory=False
)

# Menggunakan konvensi nama akademis dan memperbaiki typo koma
df.columns = ["Query", "Category", "Label"]

print(f"\n[1] DATASET DIMUAT")
print(f"    Total data mentah : {len(df)} baris")
print(f"    Kolom             : {list(df.columns)}")

print("\n[Informasi DataFrame df]")
print(f"Jumlah Baris    : {df.shape[0]} baris")
print(f"Jumlah Kolom    : {df.shape[1]} kolom")
print("\nRingkasan Informasi (df.info()):")

# JIKA Anda tetap menggunakan nama kolom 'category', gunakan baris ini:
kategori_unik = df['Category'].unique().tolist()

print("Daftar seluruh kategori yang ada:")
print(kategori_unik)
df.info()

  SISTEM DETEKSI SQL INJECTION — MULTINOMIAL NAÏVE BAYES

[1] DATASET DIMUAT
    Total data mentah : 10190450 baris
    Kolom             : ['Query', 'Category', 'Label']

[Informasi DataFrame df]
Jumlah Baris    : 10190450 baris
Jumlah Kolom    : 3 kolom

Ringkasan Informasi (df.info()):
Daftar seluruh kategori yang ada:
['None_Type', 'Error-Based', 'meta_based', 'stackqueries_based', 'Time-Based', 'Union-Based', 'boolean-based']
<class 'pandas.DataFrame'>
RangeIndex: 10190450 entries, 0 to 10190449
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   Query     str  
 1   Category  str  
 2   Label     str  
dtypes: str(3)
memory usage: 233.2 MB


In [84]:
# =============================================================================
#  TAHAP 2 — PREPROCESSING OTOMATIS
# =============================================================================

print("\n[2] PREPROCESSING OTOMATIS (ALOKASI TERBATAS)")

sebelum = len(df)

# 1. Hapus baris dengan nilai kosong
df.dropna(subset=["Query", "Label"], inplace=True)
print(f"    Hapus baris kosong       : {sebelum - len(df)} baris dihapus")

# 2. Normalisasi format Label
df["Label"] = (
    df["Label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"yes": "1", "no": "0"})
)
df = df[df["Label"].str.match(r"^[01]$")]
df["Label"] = df["Label"].astype(int)
print(f"    Setelah filter Label     : {len(df)} baris valid")

# 3. Lowercase awal untuk menangkap duplikat (Sangat Cepat)
print("    Melakukan lowercase awal pada seluruh data mentah...")
df["Query_lowercase"] = df["Query"].astype(str).str.lower()

sebelum_lowercase_dup = len(df)
df.drop_duplicates(subset=["Query_lowercase"], inplace=True)
print(f"    Duplikat berbasis lowercase dihapus: {sebelum_lowercase_dup - len(df)} baris")

print(f"    -> Mendeteksi {os.cpu_count()} Logical Processors di sistem.")
print(f"    -> Mengaktifkan {JUMLAH_JOBS} Logical Processors")

df["Query_clean"] = Parallel(n_jobs=JUMLAH_JOBS)(
    delayed(preprocess_text)(query) for query in df["Query_lowercase"]
)
print("    Preprocessing paralel selesai!")

# 5. Hapus duplikasi struktural akhir pasca-preprocessing
sebelum_struct_dup = len(df)
df.drop_duplicates(subset=["Query_clean"], inplace=True)
df = df[df["Query_clean"].str.strip() != ""]
print(f"    Duplikat struktural dihapus        : {sebelum_struct_dup - len(df)} baris")

# Hapus kolom sementara agar tidak mengotori DataFrame
df.drop(columns=["Query_lowercase"], inplace=True)

# Reset index setelah pembersihan
df.reset_index(drop=True, inplace=True)

print(f"    Total data bersih (Final)          : {len(df)} baris")
print(f"\n    Distribusi Kelas:")
distribusi = df["Label"].value_counts()
print(f"      Label 0 (Normal) : {distribusi.get(0, 0)} sampel")
print(f"      Label 1 (SQLI)   : {distribusi.get(1, 0)} sampel")


[2] PREPROCESSING OTOMATIS (ALOKASI TERBATAS)
    Hapus baris kosong       : 0 baris dihapus
    Setelah filter Label     : 10190450 baris valid
    Melakukan lowercase awal pada seluruh data mentah...
    Duplikat berbasis lowercase dihapus: 44040 baris
    -> Mendeteksi 8 Logical Processors di sistem.
    -> Mengaktifkan 8 Logical Processors
    Preprocessing paralel selesai!
    Duplikat struktural dihapus        : 8337789 baris
    Total data bersih (Final)          : 1808621 baris

    Distribusi Kelas:
      Label 0 (Normal) : 959040 sampel
      Label 1 (SQLI)   : 849581 sampel


In [85]:
# =============================================================================
#  TAHAP 3 — RINGKASAN HASIL PREPROCESSING
# =============================================================================

print("\n[3] TEXT PREPROCESSING SELESAI")
print("    Contoh hasil preprocessing:")
for i in range(min(MAX_EXAMPLES, len(df))):
    print(f"\n    [{i+1}] Asli   : {df['Query'].iloc[i][:MAX_DISPLAY_LENGTH]}")
    print(f"         Bersih : {df['Query_clean'].iloc[i][:MAX_DISPLAY_LENGTH]}")
    print(f"         Label  : {'SQLI (1)' if df['Label'].iloc[i] == 1 else 'Normal (0)'}")


[3] TEXT PREPROCESSING SELESAI
    Contoh hasil preprocessing:

    [1] Asli   : UPDATE id, email FROM login WHERE created_at LIKE '2024-01-01' OR crea
         Bersih : update id , email from login where created_at like ' str ' or created_
         Label  : Normal (0)

    [2] Asli   : UPDATE MIN(created_at) FROM payments WHERE product_id BETWEEN 1 AND pr
         Bersih : update min ( created_at ) from payments where product_id between 0 and
         Label  : Normal (0)

    [3] Asli   : EXEC created_at, updated_at FROM admin WHERE price LIKE 1 NOT price LI
         Bersih : exec created_at , updated_at from admin where price like 0 not price l
         Label  : Normal (0)

    [4] Asli   : SELECT MIN(created_at) FROM sessions WHERE id = 1 AND HAVING 1=1#
         Bersih : select min ( created_at ) from sessions where id = 0 and having 0 = 0 
         Label  : SQLI (1)

    [5] Asli   : UPDATE COUNT(*) FROM transactions WHERE product_id < 'shipped' AND pro
         Bersih : update c

In [86]:
# =============================================================================
#  TAHAP tambahan — BALANCING KELAS & DOWNSAMPLING PER KATEGORI (DINAMIS)
# =============================================================================

print("\n[tambahan] BALANCING & DOWNSAMPLING KELAS")

# Hitung ketersediaan baris unik asli pasca-generalisasi
jumlah_per_kat = {}
for kat in KATEGORI_TARGET:
    jumlah_per_kat[kat] = (df["Category"] == kat).sum()
    print(f"    Tersedia setelah deduplikasi struktural '{kat}': {jumlah_per_kat[kat]} baris")

# Optimasi Akademis: Menyesuaikan target sampling secara otomatis berdasarkan data terkecil agar tidak crash
min_tersedia = min(jumlah_per_kat.values())
TARGET_RIIL = min(TARGET_PER_KATEGORI, min_tersedia)
print(f"\n[Info] Target disesuaikan menjadi {TARGET_RIIL} baris per kategori SQLi untuk menghindari sample error.")

sampled_dfs = []

# Sampling masing-masing kategori target SQLi
for kat in KATEGORI_TARGET:
    df_kat = df[df["Category"] == kat].sample(n=TARGET_RIIL, random_state=RANDOM_STATE)
    sampled_dfs.append(df_kat)

# Menyeimbangkan data Normal (Label 0) dengan total akumulasi dari seluruh kategori SQLi yang diambil
TARGET_NORMAL = TARGET_RIIL * len(KATEGORI_TARGET)
df_normal = df[df["Label"] == 0].sample(n=TARGET_NORMAL, random_state=RANDOM_STATE)
sampled_dfs.append(df_normal)

# Gabungkan semua subset data dan acak urutannya (shuffle)
df_balanced = pd.concat(sampled_dfs, axis=0).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df = df_balanced.copy()

print(f"\n    Berhasil di-balancing berdasarkan struktur query.")
print(f"    Total data gabungan setelah balancing: {len(df)} baris")

print("\n    Distribusi kategori setelah balancing:")
print(df["Category"].value_counts())


[tambahan] BALANCING & DOWNSAMPLING KELAS
    Tersedia setelah deduplikasi struktural 'Error-Based': 71082 baris
    Tersedia setelah deduplikasi struktural 'meta_based': 334793 baris
    Tersedia setelah deduplikasi struktural 'stackqueries_based': 94207 baris
    Tersedia setelah deduplikasi struktural 'Time-Based': 291383 baris
    Tersedia setelah deduplikasi struktural 'Union-Based': 13572 baris
    Tersedia setelah deduplikasi struktural 'boolean-based': 44544 baris

[Info] Target disesuaikan menjadi 13572 baris per kategori SQLi untuk menghindari sample error.

    Berhasil di-balancing berdasarkan struktur query.
    Total data gabungan setelah balancing: 162864 baris

    Distribusi kategori setelah balancing:
Category
None_Type             81432
Error-Based           13572
Time-Based            13572
meta_based            13572
Union-Based           13572
boolean-based         13572
stackqueries_based    13572
Name: count, dtype: int64


In [87]:
# =============================================================================
#  TAHAP 4 — SPLIT DATASET (VERSI INSTAN & BERSIH)
# =============================================================================

source_df = df_balanced if "df_balanced" in globals() else df
print(f"\n[4] SPLIT DATASET ({int((1-TEST_SIZE)*100)}:{int(TEST_SIZE*100)})")

# Langsung split data asli tanpa perantara df_test_raw
df_train, df_test = train_test_split(
    source_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=source_df["Label"]
)

# Definisikan fitur dan target dari hasil split
X_train = df_train["Query_clean"].astype(str)
y_train = df_train["Label"]
X_test = df_test["Query_clean"].astype(str)
y_test = df_test["Label"]

print(f"    Data Training (X_train) : {len(X_train)} sampel")
print(f"    Data Testing (X_test)   : {len(X_test)} sampel")


[4] SPLIT DATASET (80:20)
    Data Training (X_train) : 130291 sampel
    Data Testing (X_test)   : 32573 sampel


In [107]:
# =============================================================================
#  TAHAP 5 — DEFINISI PIPELINE TF-IDF + MULTINOMIAL NAÏVE BAYES
# =============================================================================

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer=TFIDF_ANALYZER,
            ngram_range=TFIDF_NGRAM_RANGE,
            min_df=TFIDF_MIN_DF,
            max_features=TFIDF_MAX_FEATURES,
            sublinear_tf=TFIDF_SUBLINEAR_TF,
            token_pattern=TFIDF_TOKEN_PATTERN
        )
    ),
    (
        "nb",
        MultinomialNB(alpha=NB_ALPHA)
    )
])

print("\n[5] PIPELINE DIDEFINISIKAN")
print("    Algoritma  : Multinomial Naïve Bayes")
print(f"    Ekstraksi  : TF-IDF (Char n-gram {TFIDF_NGRAM_RANGE}, max {TFIDF_MAX_FEATURES} fitur)")
print(f"    Alpha (smoothing) : {NB_ALPHA}")


[5] PIPELINE DIDEFINISIKAN
    Algoritma  : Multinomial Naïve Bayes
    Ekstraksi  : TF-IDF (Char n-gram (1, 3), max 5000 fitur)
    Alpha (smoothing) : 0.1


In [108]:
# =============================================================================
#  TAHAP 6 — CROSS-VALIDATION STRATIFIED (PARALEL 3 CORES)
# =============================================================================

cv = StratifiedKFold(
    n_splits=CV_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(f"\n[6] CROSS-VALIDATION ({CV_N_SPLITS}-Fold Stratified)")
print("    Menjalankan 5-Fold Cross Validation secara paralel")

# Tambahkan n_jobs untuk membatasi thread komputasi
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=JUMLAH_JOBS
)

print(f"    F1 per Fold : {[round(s, 4) for s in scores]}")
print(f"    Rata-rata   : {scores.mean():.4f}")
print(f"    Std Dev     : {scores.std():.4f}")


[6] CROSS-VALIDATION (5-Fold Stratified)
    Menjalankan 5-Fold Cross Validation secara paralel
    F1 per Fold : [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
    Rata-rata   : 1.0000
    Std Dev     : 0.0000


In [109]:
# =============================================================================
#  TAHAP 7 — TRAINING MODEL
# =============================================================================

pipeline.fit(X_train, y_train)

print(f"\n[7] MODEL BERHASIL DILATIH")
print(f"    Jumlah data training : {len(X_train)} sampel")



[7] MODEL BERHASIL DILATIH
    Jumlah data training : 130291 sampel


In [110]:
# =============================================================================
#  TAHAP 8 — EVALUASI MODEL
# =============================================================================

y_pred = pipeline.predict(X_test)

akurasi = accuracy_score(y_test, y_pred)
presisi = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n[8] HASIL EVALUASI MODEL")
print("    " + SEPARATOR_MEDIUM)
print(f"    Akurasi   : {akurasi * 100:.2f}%")
print(f"    Presisi   : {presisi * 100:.2f}%")
print(f"    Recall    : {recall * 100:.2f}%")
print(f"    F1-Score  : {f1 * 100:.2f}%")
print("    " + SEPARATOR_MEDIUM)

tn, fp, fn, tp = cm.ravel()
print(f"\n    CONFUSION MATRIX")
print(f"    {'':20} Prediksi Normal  Prediksi SQLI")
print(f"    {'Aktual Normal':<20} {tn:<17} {fp}")
print(f"    {'Aktual SQLI':<20} {fn:<17} {tp}")
print(f"\n      TP (Benar SQLI)    : {tp}")
print(f"      TN (Benar Normal)  : {tn}")
print(f"      FP (False Positive): {fp}")
print(f"      FN (False Negative): {fn}")

print("\n    CLASSIFICATION REPORT:")
print(classification_report(
    y_test, y_pred,
    target_names=CLASSIFICATION_TARGETS
))


[8] HASIL EVALUASI MODEL
    ────────────────────────────────────────
    Akurasi   : 100.00%
    Presisi   : 100.00%
    Recall    : 100.00%
    F1-Score  : 100.00%
    ────────────────────────────────────────

    CONFUSION MATRIX
                         Prediksi Normal  Prediksi SQLI
    Aktual Normal        16287             0
    Aktual SQLI          0                 16286

      TP (Benar SQLI)    : 16286
      TN (Benar Normal)  : 16287
      FP (False Positive): 0
      FN (False Negative): 0

    CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     16287
    SQLI (1)       1.00      1.00      1.00     16286

    accuracy                           1.00     32573
   macro avg       1.00      1.00      1.00     32573
weighted avg       1.00      1.00      1.00     32573



In [114]:
# =============================================================================
#  TAHAP 9 — UJI MANUAL DETEKSI
# =============================================================================

def deteksi_sqli_dengan_waktu(input_teks: str) -> dict:
    """Fungsi deteksi SQL Injection dengan optimasi nilai ambang batas (Threshold)."""
    start_time = time.time()
    teks_bersih = preprocess_text(input_teks)
    
    # 1. Ambil nilai probabilitas murni, bukan langsung hasil prediksi prediktor
    probabilitas = pipeline.predict_proba([teks_bersih])[0]
    prob_normal = probabilitas[0]
    prob_sqli = probabilitas[1]
    
    # 2. Atur Threshold Akademis (Misal: Hanya blokir jika keyakinan SQLi >= 80%)
    # Jika model hanya 50% atau 60% yakin, anggap sebagai teks normal/kebingungan model
    THRESHOLD_BLOKIR = 0.80
    
    if prob_sqli >= THRESHOLD_BLOKIR:
        status = "🚫 DIBLOKIR"
        prediksi_label = "SQLI"
    else:
        status = "✅ DIIZINKAN"
        prediksi_label = "Normal"
        
    latency_ms = (time.time() - start_time) * 1000

    return {
        "input": input_teks,
        "prediksi": prediksi_label,
        "prob_sqli": round(prob_sqli * 100, 2),
        "prob_normal": round(prob_normal * 100, 2),
        "status": status,
        "latency_ms": round(latency_ms, 2)
    }

sampel_uji = [
    # === KATEGORI 1: DATA NORMAL (Aman / Izinkan) ===
    "admin",                                      # Kata kunci login biasa
    "buku_algoritma_pemrograman",                 # Parameter pencarian standar
    "malik.adhitya@email.com",                    # Input email biasa
    "14520",                                      # Input ID angka bersih
    "Saya mau SELECT (pilih) menu nasi goreng di WHERE (tempat) biasa", # Jebakan kata kunci SQL (Indonesian)
    "Join our student union today and select your club", # Jebakan kata kunci SQL (English)
    "Please reset my password, I cannot log in",  # Kalimat pengaduan normal
    
    # === KATEGORI 2: META-BASED & BOOLEAN-BASED SQLI (Deteksi: SQLI) ===
    "' OR '1'='1",                                # Classic Auth Bypass
    "admin' --",                                  # Simple comment bypass
    "1' OR 2+2=4 --",                             # Variasi operasi matematika
    "xyz' OR 'a'='a",                             # String comparison bypass
    
    # === KATEGORI 3: UNION-BASED SQLI (Deteksi: SQLI) ===
    "UNION SELECT username, password FROM users--",
    "-1' UNION ALL SELECT NULL, NULL, version()--", # Union dengan pemanggilan fungsi database
    "search=buku' UNION SELECT 1,2,3,4--",        # Union pada parameter pencarian
    
    # === KATEGORI 4: TIME-BASED SQLI (Deteksi: SQLI) ===
    "1' AND SLEEP(5)--",                          # Fungsi sleep standar
    "admin' AND (SELECT 1 FROM (SELECT(SLEEP(10)))x)--", # Heavy time-based query
    "1 AND BENCHMARK(5000000,MD5(1))",            # Fungsi benchmark untuk overload CPU DB
    
    # === KATEGORI 5: ERROR-BASED SQLI (Deteksi: SQLI) ===
    "1' AND EXTRACTVALUE(1, CONCAT(0x3a, VERSION()))--", # Pola XPath Error (MySQL)
    "1 AND/*!50000SELECT*/ 1",                    # Obfuscation komentar untuk memicu error/bypass
    "OR EXP(~(SELECT * FROM (SELECT 1)x))",       # Mathematical overflow error
    
    # === KATEGORI 6: STACKED QUERIES SQLI (Deteksi: SQLI) ===
    "1; DROP TABLE users;--",                     # Stacked query penghancuran table
    "123; UPDATE users SET password='123' WHERE id=1;--", # Stacked query manipulasi data
]

print("\n[9] UJI MANUAL DETEKSI")
print("    " + SEPARATOR_SHORT)

for sampel in sampel_uji:
    hasil = deteksi_sqli_dengan_waktu(sampel)
    print(f"\n    Input  : {hasil['input']}")
    print(f"    Status : {hasil['status']}")
    print(f"    P(SQLI)= {hasil['prob_sqli']}%  |  P(Normal)={hasil['prob_normal']}%")
    print(f"    Waktu  : {hasil['latency_ms']} ms  |  (Memenuhi target < 100ms)")

print("\n    " + SEPARATOR_SHORT)


[9] UJI MANUAL DETEKSI
    ────────────────────────────────────────────────────────────

    Input  : admin
    Status : ✅ DIIZINKAN
    P(SQLI)= 42.54%  |  P(Normal)=57.46%
    Waktu  : 1.73 ms  |  (Memenuhi target < 100ms)

    Input  : buku_algoritma_pemrograman
    Status : ✅ DIIZINKAN
    P(SQLI)= 50.0%  |  P(Normal)=50.0%
    Waktu  : 2.2 ms  |  (Memenuhi target < 100ms)

    Input  : malik.adhitya@email.com
    Status : ✅ DIIZINKAN
    P(SQLI)= 17.75%  |  P(Normal)=82.25%
    Waktu  : 1.02 ms  |  (Memenuhi target < 100ms)

    Input  : 14520
    Status : ✅ DIIZINKAN
    P(SQLI)= 69.17%  |  P(Normal)=30.83%
    Waktu  : 0.88 ms  |  (Memenuhi target < 100ms)

    Input  : Saya mau SELECT (pilih) menu nasi goreng di WHERE (tempat) biasa
    Status : 🚫 DIBLOKIR
    P(SQLI)= 99.83%  |  P(Normal)=0.17%
    Waktu  : 0.93 ms  |  (Memenuhi target < 100ms)

    Input  : Join our student union today and select your club
    Status : 🚫 DIBLOKIR
    P(SQLI)= 99.96%  |  P(Normal)=0.04%
    W

In [93]:
# =============================================================================
#  TAHAP 10 — SIMPAN MODEL
# =============================================================================

joblib.dump(pipeline, MODEL_PATH)

print(f"\n[10] MODEL DISIMPAN")
print(f"    Path  : {MODEL_PATH}")
print(f"    Muat kembali dengan: pipeline = joblib.load('{MODEL_PATH}')")
print("\n" + SEPARATOR_LONG)
print("  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK")
print(SEPARATOR_LONG)


[10] MODEL DISIMPAN
    Path  : model_sqli_nb.pkl
    Muat kembali dengan: pipeline = joblib.load('model_sqli_nb.pkl')

  PROSES SELESAI — MODEL SIAP UNTUK DEPLOYMENT KE FLASK
